# Assignment 1 – Problem 3
## Satellite Image Classification (Urban vs. Natural) Using Classical Vision + ML  
### Classical Computer Vision + Machine Learning
**Course:** Computer Vision (S2-25_AIMLCZG525)

---

### Goal 
Build a classical machine-learning system that classifies satellite images into Urban or 
Natural categories using handcrafted image features. The system must incorporate pixel
level, texture, and gradient-based features and evaluate performance on a real external 
image. 

### Implementation Overview
1. Dataset Preparation (fMoW dataset – Urban vs Natural, ~500 each)
2. Preprocessing (resize, convert to grayscale, CLAHE, Gaussian blur)
3. Feature Engineering (Pixel intensity histogram, LBP texture descriptor, Edges using Sobel/Canny, Histogram of Oriented Gradients (HOG))
4. Feature Selection (PCA)
5. Model Training (SVM + Random Forest, 5-fold CV)
6. Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)
7. External Image Prediction
8. Analysis & Discussion

## 1. Dataset Preparation
**fMoW Dataset – Urban vs Natural (~500 images each)**

- Download and organize fMoW dataset images into two classes: Urban and Natural
- Limit to ~500 images per class for balanced training
- Split into train/validation/test sets

In [ ]:
# 1. Dataset Preparation


## 2. Preprocessing
**Steps: Resize → Grayscale → CLAHE → Gaussian Blur**

- Resize all images to a fixed resolution (e.g., 128×128)
- Convert to grayscale for uniform feature extraction
- Apply CLAHE (Contrast Limited Adaptive Histogram Equalization)
- Apply Gaussian blur to reduce noise

In [ ]:
# 2. Preprocessing


## 3. Feature Engineering
**Descriptors: Pixel Intensity Histogram | LBP | Sobel/Canny Edges | HOG**

- **Pixel Intensity Histogram**: Distribution of pixel values per channel
- **LBP (Local Binary Pattern)**: Texture descriptor capturing local structure
- **Edges (Sobel / Canny)**: Edge magnitude maps for structural features
- **HOG (Histogram of Oriented Gradients)**: Shape and gradient orientation features

In [ ]:
# 3. Feature Engineering


## 4. Feature Selection
**Method: Principal Component Analysis (PCA)**

- Concatenate all engineered features into a single feature vector
- Apply PCA to reduce dimensionality while retaining variance (e.g., 95%)
- Visualize explained variance ratio

In [ ]:
# 4. Feature Selection (PCA)


## 5. Model Training
**Classifiers: SVM + Random Forest | Validation: 5-Fold Cross-Validation**

- Train Support Vector Machine (SVM) with RBF kernel on PCA-reduced features
- Train Random Forest classifier with tuned hyperparameters
- Apply 5-fold cross-validation to assess generalization performance

In [ ]:
# 5. Model Training (SVM + Random Forest, 5-Fold CV)

# ──────────────────────────────────────────────────────────────
# Define four classical ML classifiers as required by the assignment
# ──────────────────────────────────────────────────────────────
classifiers = {
    'k-NN (k=5)': KNeighborsClassifier(
        n_neighbors=5,
        metric='euclidean',
        n_jobs=-1
    ),
    'SVM (RBF)': SVC(
        kernel='rbf',
        C=10,
        gamma='scale',
        decision_function_shape='ovr',
        random_state=SEED
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        n_jobs=-1,
        random_state=SEED
    ),
    'Logistic Regression': LogisticRegression(
        max_iter=500,
        solver='lbfgs',
        multi_class='multinomial',
        C=1.0,
        random_state=SEED
    )
}

print('✅ Classifiers defined:')
for name in classifiers:
    print(f'   • {name}')


# ── 5-fold Stratified Cross-Validation on training set ─────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_results = {}
print('Running 5-fold cross-validation …\n')

for name, clf in classifiers.items():
    scores = cross_val_score(clf, X_tr, y_train,
                              cv=skf, scoring='accuracy', n_jobs=-1)
    cv_results[name] = scores
    print(f'  {name:<25} CV Accuracy: '
          f'{scores.mean():.4f} ± {scores.std():.4f}')

# ── CV Results Bar Chart ─────────────────────────────────────
names  = list(cv_results.keys())
means  = [cv_results[n].mean() for n in names]
stds   = [cv_results[n].std()  for n in names]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(names, means, yerr=stds, capsize=6,
              color=['#4C72B0','#DD8452','#55A868','#C44E52'],
              edgecolor='black', linewidth=0.6)
ax.set_ylim(0, 1.0)
ax.set_ylabel('CV Accuracy', fontsize=11)
ax.set_title('5-Fold Cross-Validation Accuracy per Model', fontweight='bold')
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, mean + 0.01,
            f'{mean:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()


import time

trained_models = {}
train_times    = {}

print('Training classifiers on full training set …\n')
for name, clf in classifiers.items():
    t0 = time.time()
    clf.fit(X_tr, y_train)
    elapsed = time.time() - t0
    trained_models[name] = clf
    train_times[name]    = elapsed
    print(f'  ✅ {name:<25} trained in {elapsed:.2f}s')

print('\nAll models trained.')

## 6. Evaluation
**Metrics: Accuracy | Precision | Recall | F1-Score | Confusion Matrix**

- Compute Accuracy, Precision, Recall, and F1-Score for each model
- Plot Confusion Matrix for SVM and Random Forest
- Compare performance of both classifiers

In [ ]:
# 6. Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)

eval_results = {}

print(f'{'Model':<25} {'Accuracy':>10} {'Precision':>11} {'Recall':>8} {'F1-Score':>10}')
print('-' * 70)

for name, clf in trained_models.items():
    y_pred = clf.predict(X_te)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    eval_results[name] = {
        'accuracy' : acc,
        'precision': prec,
        'recall'   : rec,
        'f1'       : f1,
        'y_pred'   : y_pred
    }
    print(f'{name:<25} {acc:>10.4f} {prec:>11.4f} {rec:>8.4f} {f1:>10.4f}')

# ── Best model ──────────────────────────────────────────────
best_name = max(eval_results, key=lambda k: eval_results[k]['f1'])
print(f'\n🏆 Best model (by F1): {best_name}  '
      f'(F1 = {eval_results[best_name]["f1"]:.4f})')


## 7. External Image Prediction
**Predict class (Urban / Natural) for a new unseen image**

- Load an external satellite image not seen during training
- Apply the same preprocessing and feature extraction pipeline
- Predict and display the class label using the best trained model

In [ ]:
# 7. External Image Prediction


## 8. Analysis & Discussion
**Observations, Insights, and Conclusions**

- Discuss which features contributed most to classification performance
- Compare SVM vs Random Forest: strengths and weaknesses
- Analyze where the model succeeds and where it fails (error analysis)
- Suggest possible improvements (e.g., deep features, data augmentation)
- Final conclusions and takeaways

In [ ]:
# 8. Analysis & Discussion
